In [1]:
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_agentchat.messages import UserMessage

## TODO: Local Model Calling.

model_client = OllamaChatCompletionClient(model="qwen2.5:14b")

In [2]:
input = [UserMessage(content="Assalamualikum, how are you!!", source="user")]
response = await model_client.create(input)
print(response.content)

Waalaikumsalam! How can I assist you today?


# **Memory in `AutoGen`**

In [3]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_core.memory import ListMemory, MemoryContent, MemoryMimeType

In [7]:
# Initialize user memory
user_memory = ListMemory()

# Add user preferences to memory
await user_memory.add(MemoryContent(content="The weather should be in metric units", mime_type=MemoryMimeType.TEXT))

await user_memory.add(MemoryContent(content="Meal recipe must be vegan", mime_type=MemoryMimeType.TEXT))

await user_memory.add(MemoryContent(content="User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.", mime_type=MemoryMimeType.TEXT))


async def get_weather(city: str, units: str = "imperial") -> str:
    if units == "imperial":
        return f"The weather in {city} is 73 °F and Sunny."
    elif units == "metric":
        return f"The weather in {city} is 23 °C and Sunny."
    else:
        return f"Sorry, I don't know the weather in {city}."


assistant_agent = AssistantAgent(
    name="assistant_agent",
    model_client=OllamaChatCompletionClient(model="qwen2.5:14b"),
    tools=[get_weather],
    memory=[user_memory],
)


In [8]:
# Run the agent with a task.
stream = assistant_agent.run_stream(task="Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.")
await Console(stream)


---------- TextMessage (user) ----------
Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.
---------- MemoryQueryEvent (assistant_agent) ----------
[MemoryContent(content='The weather should be in metric units', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='Meal recipe must be vegan', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None)]
---------- TextMessage (assistant_agent) ----------
I don't have specific information about places to hang out in Bashundhara, Dhaka that match the names "restreamnet" or "spatial place," which might be misspellings. However, I can suggest a few popular spots for hanging out with friends:

1. **Bashundhara Shopping Complex** - A

TaskResult(messages=[TextMessage(id='69eba3ca-b0f9-47fb-950e-289c9e1cd22b', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 5, 4, 23, 54, 199679, tzinfo=datetime.timezone.utc), content="Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.", type='TextMessage'), MemoryQueryEvent(id='0de09501-8838-4586-8e97-ea3ca0a6418c', source='assistant_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 5, 4, 23, 54, 206032, tzinfo=datetime.timezone.utc), content=[MemoryContent(content='The weather should be in metric units', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='Meal recipe must be vegan', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.', mime_type=<MemoryMimeType.TE

In [9]:
# Run the agent with a task.
stream = assistant_agent.run_stream(task="What is the weather in Dhaka?")
await Console(stream)


---------- TextMessage (user) ----------
What is the weather in Dhaka?
---------- MemoryQueryEvent (assistant_agent) ----------
[MemoryContent(content='The weather should be in metric units', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='Meal recipe must be vegan', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None)]
---------- ToolCallRequestEvent (assistant_agent) ----------
[FunctionCall(id='0', arguments='{"city": "Dhaka", "units": "metric"}', name='get_weather')]
---------- ToolCallExecutionEvent (assistant_agent) ----------
[FunctionExecutionResult(content='The weather in Dhaka is 23 °C and Sunny.', name='get_weather', call_id='0', is_error=False)]
---------- ToolCallSummaryMessage (assistant_agent) ----------
The weather in Dhaka is 23 °C and Sun

TaskResult(messages=[TextMessage(id='e4374ca3-ade2-4b86-ba43-b9d19faf4a8c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 5, 4, 25, 56, 968409, tzinfo=datetime.timezone.utc), content='What is the weather in Dhaka?', type='TextMessage'), MemoryQueryEvent(id='f3dba6d6-3482-45d3-8634-e75b0a22072c', source='assistant_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 5, 4, 25, 56, 972585, tzinfo=datetime.timezone.utc), content=[MemoryContent(content='The weather should be in metric units', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='Meal recipe must be vegan', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None), MemoryContent(content='User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.', mime_type=<MemoryMimeType.TEXT: 'text/plain'>, metadata=None)], type='MemoryQueryEvent'), ToolCallRequestEvent(id='bc9c16f0-f778-4c63-8ff7-b32

In [19]:
all_messages = await assistant_agent.model_context.get_messages()

all_messages

[UserMessage(content="Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.", source='user', type='UserMessage'),
 SystemMessage(content='\nRelevant memory content (in chronological order):\n1. The weather should be in metric units\n2. Meal recipe must be vegan\n3. User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.\n', type='SystemMessage'),
 AssistantMessage(content='I don\'t have specific information about places to hang out in Bashundhara, Dhaka that match the names "restreamnet" or "spatial place," which might be misspellings. However, I can suggest a few popular spots for hanging out with friends:\n\n1. **Bashundhara Shopping Complex** - A large shopping mall with various restaurants and cafes.\n2. **Gulshan Lake Restaurant & Cafe** - Offers an outdoor seating area by the lake for relaxation.\n3. **Mymensingh Road (Road No 9)** - Known for its s

# **Custom Memory Stores (Vector DBs, etc.)**

### ***Chroma db***

You can build on the `Memory` protocol to implement more complex memory stores. For example, you could implement a custom memory store that uses a vector database to store and retrieve information, or a memory store that uses a machine learning model to generate personalized responses based on the user’s preferences etc.

Specifically, you will need to overload the `add`, `query` and `update_context` methods to implement the desired functionality and pass the memory store to your agent.

Currently the following example memory stores are available as part of the `autogen_ext` extensions package.

- `autogen_ext.memory.chromadb.ChromaDBVectorMemory`: A memory store that uses a vector database to store and retrieve information.

- `autogen_ext.memory.chromadb.SentenceTransformerEmbeddingFunctionConfig`: A configuration class for the SentenceTransformer embedding function used by the `ChromaDBVectorMemory` store. 

    - Note that other embedding functions such as `autogen_ext.memory.openai.OpenAIEmbeddingFunctionConfig` can also be used with the `ChromaDBVectorMemory` store.

- `autogen_ext.memory.redis_memory.RedisMemory`: A memory store that uses a Redis vector database to store and retrieve information.

In [1]:
import tempfile

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_core.memory import MemoryContent, MemoryMimeType
from autogen_ext.memory.chromadb import (
    ChromaDBVectorMemory,
    PersistentChromaDBVectorMemoryConfig,
    SentenceTransformerEmbeddingFunctionConfig,
)
from autogen_ext.models.ollama import OllamaChatCompletionClient

In [3]:
with tempfile.TemporaryDirectory() as tmpdir:
    chroma_user_memory = ChromaDBVectorMemory(
        config=PersistentChromaDBVectorMemoryConfig(
            collection_name="preferences",
            persistence_path=tmpdir,  # Use the temp directory here
            k=2,  # Return top k results
            score_threshold=0.4,  # Minimum similarity score
            embedding_function_config=SentenceTransformerEmbeddingFunctionConfig(
                model_name="all-MiniLM-L6-v2"  # Use default model for testing
            ),
        )
    )

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [5]:
 # Add user preferences to memory
await chroma_user_memory.add(
    MemoryContent(
        content="The weather should be in metric units",
        mime_type=MemoryMimeType.TEXT,
        metadata={"category": "preferences", "type": "units"},
    )
)

await chroma_user_memory.add(
    MemoryContent(
        content="Meal recipe must be vegan",
        mime_type=MemoryMimeType.TEXT,
        metadata={"category": "preferences", "type": "dietary"},
    )
)

await chroma_user_memory.add(
    MemoryContent(
        content="User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.",
        mime_type=MemoryMimeType.TEXT,
        metadata={"category": "preferences", "type": "dietary"},
    )
)

c:\Users\Lenovo\anaconda3\envs\lang_aca\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **Build the agents**

In [3]:
async def get_weather(city: str, units: str = "imperial") -> str:
    if units == "imperial":
        return f"The weather in {city} is 73 °F and Sunny."
    elif units == "metric":
        return f"The weather in {city} is 23 °C and Sunny."
    else:
        return f"Sorry, I don't know the weather in {city}."

In [8]:
model_client = OllamaChatCompletionClient(model="qwen2.5:14b")

assistant_agent2 = AssistantAgent(
    name="assistant_agent2",
    model_client=model_client,
    tools = [get_weather],
    memory=[chroma_user_memory]
)

## **Run the agent**

In [9]:
# Run the agent with a task.
stream = assistant_agent2.run_stream(task="Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.")
await Console(stream)

await model_client.close()
await chroma_user_memory.close()

---------- TextMessage (user) ----------
Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.
---------- TextMessage (assistant_agent2) ----------
It seems like there might have been a typo in your message, but I understand that you're looking for recommendations on places to hang out with friends, similar to restaurants or venues known for social gatherings. Could you please specify which city you are referring to? This will help me provide more accurate suggestions based on the local options available.

Also, could you clarify if there is a specific type of place you prefer (e.g., café, bar, restaurant, park)?


In [12]:
chroma_user_memory.dump_component().model_dump()

{'provider': 'autogen_ext.memory.chromadb.ChromaDBVectorMemory',
 'component_type': 'memory',
 'version': 1,
 'component_version': 1,
 'description': 'Store and retrieve memory using vector similarity search powered by ChromaDB.',
 'label': 'ChromaDBVectorMemory',
 'config': {'client_type': 'persistent',
  'collection_name': 'preferences',
  'distance_metric': 'cosine',
  'k': 2,
  'score_threshold': 0.4,
  'allow_reset': False,
  'tenant': 'default_tenant',
  'database': 'default_database',
  'embedding_function_config': {'function_type': 'sentence_transformer',
   'model_name': 'all-MiniLM-L6-v2'},
  'persistence_path': 'C:\\Users\\Lenovo\\AppData\\Local\\Temp\\tmpo_lcq6qt'}}

# **Mem0Memory Example**

`autogen_ext.memory.mem0.Mem0Memory` provides integration with Mem0.ai’s memory system. It supports both cloud-based and local backends, offering advanced memory capabilities for agents. The implementation handles proper retrieval and context updating, making it suitable for production environments.

In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_core.memory import MemoryContent, MemoryMimeType
from autogen_ext.memory.mem0 import Mem0Memory
from autogen_ext.models.ollama import OllamaChatCompletionClient
import os
from dotenv import load_dotenv
load_dotenv()

mem0_api_key = os.getenv("MEM0_API_KEY")

In [23]:
model_client = OllamaChatCompletionClient(model="qwen2.5:14b")

mem0_memory = Mem0Memory(
    user_id="user_alamin",
    is_cloud=True,
    api_key = mem0_api_key
)

In [ ]:
try:
    print("Attempt to add memory in the Cloud.")
    messages = [
        {
            "role": "user",
            "content": "i'm a programer and love to build agent applications.",
        },
        {
            "role": "assistant",
            "content": "Grate to hare that you are a gen ai developer. i'll remember it you are love to build agent.",
        }
    ]

    mem0_memory._client.add(messages, user_id="user_alamin", metadata={"category": "preferences", "type": "programmer"})
    messages = [
        {
            "role": "user",
            "content": "User name is Al Amin and his university is North South University. He lives in Bashundhara, Dhaka.",
        },
        {
            "role": "assistant",
            "content": "Grate your name is Al Amin and your university is North South University and you live in bashundhara, dhaka which a grate city.",
        }
    ]

    mem0_memory._client.add(messages, user_id="user_alamin", metadata={"category": "preferences", "type": "user_info"})

except Exception as e:
    print(e)

Attempt to add memory in the Cloud.


c:\Users\Lenovo\anaconda3\envs\lang_aca\Lib\site-packages\mem0\client\utils.py:18: DeprecationWarning: output_format='v1.0' is deprecated therefore setting it to 'v1.1' by default. Check out the docs for more information: https://docs.mem0.ai/platform/quickstart#4-1-create-memories
  return func(*args, **kwargs)


## **Build the Agent**

In [25]:
agent = AssistantAgent(
    name="assistant",
    model_client=model_client,
    memory=[mem0_memory],
    system_message="Your are a helpful assistant that remember user preferences and use them to provide personalized response."
)

In [30]:
async def main():
    try:
        stream = agent.run_stream(task="Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.")
        await Console(stream)

    except Exception as e:
        print(e)

In [31]:
await main()

---------- TextMessage (user) ----------
Tomorrow some friend come in my home and i think i'll hangout with friend. can you recommend some good place like restreamnet or spatial place.


---------- MemoryQueryEvent (assistant) ----------
[MemoryContent(content='Lives in Bashundhara, Dhaka', mime_type='text/plain', metadata={'type': 'user_info', 'category': 'preferences', 'score': 0.38219040632247925, 'created_at': datetime.datetime(2025, 8, 5, 1, 49, 23, 190114, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))), 'updated_at': datetime.datetime(2025, 8, 5, 1, 49, 23, 212363, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))), 'categories': ['personal_details']}), MemoryContent(content='Loves to build agent applications', mime_type='text/plain', metadata={'type': 'programmer', 'category': 'preferences', 'score': 0.35406390702343604, 'created_at': datetime.datetime(2025, 8, 5, 1, 49, 9, 787274, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))), 'updated_at': datetime.datetime(2025, 8, 5, 1, 49, 9, 806453, tzinfo=datetime.timezone(datetime.timedelta(days=-1, seconds=61200))), 'categories': ['technology']}), MemoryC